# What CLTK knows about a Latin sentence

Two questions:

1. **How much can CLTK tell us about a single sentence?** One passage, put
   through every Latin component that works offline.
2. **Which of our eighteen operations could CLTK help detect?** Six are
   currently `detectable = False`. This notebook checks, one by one, whether
   CLTK closes any of those gaps.

Today the pipeline uses exactly two things from CLTK — `LatinBackoffLemmatizer`
for lemmas and `CollatinusDecliner` for inflection, both in
`retexo/resources/morphology.py`. Everything else here is unused.

> Some components need a corpus download and are marked. They are wrapped in
> `try` so the notebook runs end to end without network access.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import cltk
print("cltk", cltk.__version__)

SENTENCE = "arma virumque cano Troiae qui primus ab oris Italiam fato profugus"
print("sentence:", SENTENCE)

cltk cltk 1.1.7
sentence: arma virumque cano Troiae qui primus ab oris Italiam fato profugus


# Part 1 — Everything CLTK will tell us about this sentence

## 1.1 Orthography and normalization

Latin editions vary in ways that are the *editor's* choice, not the author's:
u/v and i/j, macrons, ligatures, editorial brackets. Normalizing these is the
precondition for `NOP` — two tokens count as identical only after this step.

We already do this ourselves in `retexo/normalize.py`, which additionally
folds Greek-derived spellings (*th*/*t*, *ph*/*p*, *ch*/*c*, *y*/*i*). CLTK's
helpers are narrower but well tested.

In [2]:
from cltk.alphabet.lat import (
    JVReplacer, remove_macrons, remove_accents, normalize_lat, drop_latin_punctuation,
)

jv = JVReplacer()
print("original     :", SENTENCE)
print("j/v folded   :", jv.replace(SENTENCE))
print("macrons off  :", remove_macrons("arma virumque canō Trōiae"))
print("normalize_lat:", normalize_lat("arma virumque cano", drop_accents=True, drop_macrons=True))
print()
print("our own normalize(), for comparison:")
import sys; sys.path.insert(0, ".")
from retexo.core.normalize import normalize
for token in ["uox", "vox", "Troiae", "philosophia", "rhetor"]:
    print(f"   {token:<14} -> {normalize(token)}")

original     : arma virumque cano Troiae qui primus ab oris Italiam fato profugus
j/v folded   : arma uirumque cano Troiae qui primus ab oris Italiam fato profugus
macrons off  : arma virumque cano Troiae
normalize_lat: arma virumque cano

our own normalize(), for comparison:
   uox            -> uox
   vox            -> uox
   Troiae         -> troiae
   philosophia    -> pilosopia
   rhetor         -> retor


## 1.2 Word tokenization — and a real problem with enclitics

Latin glues enclitics onto their host word: *virum* + *-que* becomes
*virumque*. Splitting them is exactly our `SPLIT` operation, so CLTK's
tokenizer is the obvious candidate.

It advertises the feature. Its own docstring promises
`abuterque -> ['abuter', '-que']`. **In version 1.1.7 that does not happen** —
run the cell and compare against the docstring.

In [3]:
from cltk.tokenizers.lat.lat import LatinWordTokenizer

tokenizer = LatinWordTokenizer()
print("our sentence :", tokenizer.tokenize(SENTENCE))
print()
print("the tokenizer's own docstring example:")
print("   promised :", ["atque", "haec", "abuter", "-que", "puer", "-ve", "pater", "-ne", "nihil"])
print("   actual   :", tokenizer.tokenize("atque haec abuterque puerve paterne nihil"))
print()
for word in ["virumque", "Lauiniaque", "litoraque", "bonusne", "amatust"]:
    print(f"   {word:<12} -> {tokenizer.tokenize(word)}")

our sentence : ['arma', 'virumque', 'cano', 'Troiae', 'qui', 'primus', 'ab', 'oris', 'Italiam', 'fato', 'profugus']

the tokenizer's own docstring example:
   promised : ['atque', 'haec', 'abuter', '-que', 'puer', '-ve', 'pater', '-ne', 'nihil']
   actual   : ['atque', 'haec', 'abuterque', 'puerve', 'paterne', 'nihil']

   virumque     -> ['virumque']
   Lauiniaque   -> ['Lauiniaque']
   litoraque    -> ['litoraque']
   bonusne      -> ['bonus', '-ne']
   amatust      -> ['amatus', 'est']


**What is going on.** Reading `LatinWordTokenizer.tokenize`, the enclitic loop
has a branch per enclitic. `n` and `st` split properly. Everything else —
`que`, `ne`, `ue`, `ve` — falls to an `else` that appends the token
**unchanged** while still setting `is_enclitic = True`:

```python
else:
    specific_tokens += [token]
    is_enclitic = True
    break
```

So the enclitic is *detected* and then not split. `bonusne` only splits because
of a separate special case that applies to the first token of a sentence.

**Consequence for us:** CLTK will not give us enclitic `SPLIT` detection out of
the box. The good news is that this is the easiest of our undetectable
operations to implement ourselves — a suffix check plus "is the remainder a
known word", which the lemmatizer can answer.

In [4]:
# The check CLTK is not doing, in three lines.
from cltk.lemmatize.lat import LatinBackoffLemmatizer

lemmatizer = LatinBackoffLemmatizer()
ENCLITICS = ("que", "ve", "ne")


def split_enclitic(token):
    """Return (stem, enclitic) if the token plausibly carries one."""
    for enclitic in ENCLITICS:
        if token.lower().endswith(enclitic) and len(token) > len(enclitic) + 2:
            stem = token[: -len(enclitic)]
            lemma = lemmatizer.lemmatize([stem])[0][1]
            if lemma and lemma.lower() != stem.lower():   # the stem is a known word
                return stem, enclitic
    return None


for word in ["virumque", "Lauiniaque", "litoraque", "neque", "denique"]:
    print(f"   {word:<12} -> {split_enclitic(word)}")

   virumque     -> ('virum', 'que')
   Lauiniaque   -> ('Lauinia', 'que')
   litoraque    -> ('litora', 'que')
   neque        -> None
   denique      -> None


## 1.3 Sentence segmentation

Relevant to `DISPERSE`, which is defined over a **clause** boundary. CLTK gives
us *sentence* boundaries — clauses would need a dependency parse.

In [5]:
from cltk.sentence.lat import LatinPunktSentenceTokenizer

try:
    splitter = LatinPunktSentenceTokenizer()
    text = "Arma virumque cano. Troiae qui primus ab oris Italiam venit. Multum ille iactatus est."
    for i, sentence in enumerate(splitter.tokenize(text)):
        print(f"   {i}: {sentence}")
except Exception as exc:
    print("needs a download:", type(exc).__name__, exc)

   0: Arma virumque cano.
   1: Troiae qui primus ab oris Italiam venit.
   2: Multum ille iactatus est.


## 1.4 Lemmatization — what backs `MORPH`

The one component doing real work in our pipeline today. `MORPH` fires when two
tokens share a lemma.

In [6]:
tokens = SENTENCE.split()
print(f"{'token':<12} {'lemma':<12}")
for token, lemma in lemmatizer.lemmatize(tokens):
    flag = "" if lemma != token else "   (unchanged)"
    print(f"{token:<12} {lemma:<12}{flag}")

token        lemma       
arma         arma           (unchanged)
virumque     vir         
cano         cano           (unchanged)
Troiae       Troius      
qui          qui            (unchanged)
primus       primus         (unchanged)
ab           ab             (unchanged)
oris         ora         
Italiam      Italia      
fato         fatum       
profugus     profugus       (unchanged)


Note `Troiae -> Troius` — the lemmatizer picks an adjectival lemma where the
context wants the place name *Troia*. Backoff lemmatization has no access to
syntax, so ambiguous forms resolve by frequency rather than by context. This is
a quiet source of missed `MORPH` links.

## 1.5 Morphology — generation, not analysis

An important limitation. `CollatinusDecliner` **generates** every inflected
form of a lemma. It does not *analyse* a surface form into its features.

So we can ask "what are all the forms of *arma*?" but not "what case is
*armis*?". That is why our `MORPH` detection is lemma-equality only, and why a
script says *that* the inflection changed but never *how*. Real morphological
analysis needs the Stanza pipeline (§1.9).

In [7]:
from cltk.morphology.lat import CollatinusDecliner

decliner = CollatinusDecliner()
forms = decliner.decline("arma")
print(f"{len(forms)} forms of 'arma'; first 8 (form, morpho-tag):")
for form, tag in forms[:8]:
    print(f"   {form:<12} {tag}")

print()
print("What we cannot do: given 'armis', ask for its case and number.")
print("The workaround is to invert the table --")
surface = "armis"
matches = [tag for form, tag in forms if form == surface]
print(f"   {surface} matches tags: {matches}")

6 forms of 'arma'; first 8 (form, morpho-tag):
   arma         --p----n-
   arma         --p----v-
   arma         --p----a-
   armorum      --p----g-
   armis        --p----d-
   armis        --p----b-

What we cannot do: given 'armis', ask for its case and number.
The workaround is to invert the table --
   armis matches tags: ['--p----d-', '--p----b-']


## 1.6 Stopwords — what `ADAPT` would need

`ADAPT` marks a region where substitutions concentrate, which requires telling
content words from function words.

In [8]:
from cltk.stops.lat import STOPS

print(f"CLTK Latin stoplist: {len(STOPS)} entries")
print("sample:", STOPS[:12])
print()
for probe in ["quippe", "interdum", "autem", "igitur"]:
    print(f"   {probe:<10} in stoplist: {probe in STOPS}")

CLTK Latin stoplist: 92 entries
sample: ['ab', 'ac', 'ad', 'adhic', 'aliqui', 'aliquis', 'an', 'ante', 'apud', 'at', 'atque', 'aut']

   quippe     in stoplist: False
   interdum   in stoplist: False
   autem      in stoplist: True
   igitur     in stoplist: True


**Verdict, already reached in `retexo/substitution.py`:** 92 entries is not
enough, and the list contains neither *quippe* nor *interdum* — both of which a
contextual model will happily rank first in any slot. We exclude function words
by corpus *frequency* instead, keeping the list only as a floor. This notebook
confirms that decision rather than changing it.

## 1.7 Syllables, scansion and phonology

Not currently used by any operation. Worth knowing about: metrical position is
a real constraint on poetic reuse — a poet substituting a word usually needs
one that scans — so this could support a future operation, or a filter on
generated substitutions.

In [9]:
from cltk.prosody.lat.scanner import Syllabifier
from cltk.phonology.lat.transcription import Transcriber

print("syllables :", Syllabifier().syllabify("arma virumque cano Troiae"))
print("IPA       :", Transcriber("Classical", "Allen").transcribe("arma virumque cano"))

syllables : ['ar', 'ma', 'vi', 'rum', 'que', 'ca', 'no', 'Tro', 'i', 'ae']


IPA       : ['ar.ma wɪ.'rʊm.kʷɛ 'ka.n̪ɔ]


## 1.8 Named entities — the finding that matters

`NE-SUB` is currently `detectable = False`, with this reason in
`operations/token.py`:

> Deciding that two proper nouns stand in for one another needs a named-entity
> pass, and the capitalization test the earlier prototype used fires on any
> sentence-initial word, since capitalization in Latin editions is editorial.

**CLTK ships that named-entity pass**, offline, as a list of roughly 39,000 Latin
proper names.

In [10]:
from cltk.tag.ner import tag_ner

tagged = tag_ner("lat", input_text="Gallia est omnis divisa in partes tres Caesar Roma Troia")
for item in tagged:
    mark = "ENTITY" if len(item) > 1 else ""
    print(f"   {item[0]:<10} {mark}")

print()
import os
names_path = os.path.expanduser("~/cltk_data/lat/model/lat_models_cltk/ner/proper_names.txt")
names = {line.strip() for line in open(names_path, encoding="utf-8") if line.strip()}
print(f"proper-name list: {len(names):,} entries")
for probe in ["Troia", "Italia", "Hesperia", "Maro", "Aeneas", "Vergilius"]:
    print(f"   {probe:<12} known: {probe in names}")

   Gallia     ENTITY
   est        
   omnis      
   divisa     
   in         
   partes     
   tres       
   Caesar     ENTITY
   Roma       ENTITY
   Troia      ENTITY

proper-name list: 38,976 entries
   Troia        known: True
   Italia       known: True
   Hesperia     known: True
   Maro         known: True
   Aeneas       known: True
   Vergilius    known: False


Every name our `NE-SUB` demonstration uses — *Troia*, *Italia*, *Hesperia* — is
in the list. The stated blocker for `NE-SUB` detection is therefore **already
solved by a dependency we install anyway.**

The remaining work is not recognition but *pairing*: knowing both tokens are
entities does not establish that one stands in for the other. That still needs
a similarity signal or a gazetteer of alternative names — but it is a much
smaller problem than the one recorded in the docstring.

## 1.9 What needs a download

Two components we could not exercise offline. Both are worth knowing about.

- **`LatinLewisLexicon`** — Lewis's *Elementary Latin Dictionary* (1890) as
  queryable glosses. A gloss-overlap test is a cheap, resource-light proxy for
  synonymy where WordNet is silent.
- **The Stanza pipeline** (`LatinStanzaProcess`) — real morphological
  *analysis* plus a dependency parse. This is what §1.5 lacks, and the only
  route to clause boundaries for `DISPERSE`.

In [11]:
# Both of these prompt for a download the first time; run them yourself if wanted.
try:
    from cltk.lexicon.lat import LatinLewisLexicon
    print(LatinLewisLexicon().lookup("arma")[:200])
except Exception as exc:
    print("Lewis lexicon not available offline:", type(exc).__name__)

from cltk.languages.pipelines import LatinPipeline
print()
print("Latin pipeline processes:")
for process in LatinPipeline().processes:
    print("   -", process.__name__)

arma ōrum, n 1 AR-, implements, outfit, instruments, tools: cerealia, for making bread, V.: (coloni) operis, O.: omne genus: armorum, Cs.: Conligere arma iubet, the ship's tackle, V.—Armor fitted to t

Latin pipeline processes:
   - LatinNormalizeProcess
   - LatinStanzaProcess
   - LatinEmbeddingsProcess
   - StopsProcess
   - LatinLexiconProcess


## 1.10 WordNet through CLTK

CLTK wraps the same Exeter Latin WordNet our oracle already queries directly,
so this is an alternative interface rather than new data — and it inherits the
same ~30% lemma coverage that drives most of our INS/DEL labels.

In [12]:
from cltk.wordnet.wordnet import WordNetCorpusReader

latin_wordnet = WordNetCorpusReader(iso_code="lat")

for lemma in latin_wordnet.lemmas("arma", "n"):
    print("lemma :", lemma.lemma(), f"({lemma.pos()})")
    for synset in list(lemma.synsets())[:2]:
        print("   gloss   :", synset.gloss())
        print("   hypernyms:", [s.gloss()[:40] for s in list(synset.hypernyms())[:2]])

lemma : arma (n)


   gloss   : the weaponry available for the defense of a region


   hypernyms: ['weapons considered collectively']
   gloss   : weaponry used in fighting or hunting


   hypernyms: ['weapons considered collectively']


### Derivational relations — and a correction

`Lemma` exposes `derivationally_related_forms()`. That is precisely what `POS`
needs, and it contradicts what `operations/token.py` currently claims:

> A true derivational pair crosses lemmas — *amāre* to *amor* — so
> lemmatization cannot find it, and no derivational lexicon for Latin has been
> identified.

The first half is right; the second is not. Latin WordNet carries derivational
links, they are reachable through the resource our oracle already queries, and
**the docstring's own example resolves**: `amo` does list `amor`.

*(Note `lemma.lemma` is a **method**, not a property — calling it is easy to
get wrong and yields bound-method repr strings instead of words.)*

In [13]:
def derivations(word, pos):
    """Derivationally related forms of a lemma, as plain strings."""
    entries = list(latin_wordnet.lemmas(word, pos))
    if not entries:
        return None
    return sorted({l.lemma() for l in entries[0].derivationally_related_forms()})


for word, pos in [("cano", "v"), ("fugio", "v"), ("bellum", "n"), ("amo", "v")]:
    found = derivations(word, pos)
    print(f"{word:<8} ({pos}) -> {found[:7] if found else 'no lemma'}")

print()
print("The poetically relevant one:")
print("   cano -> carmen :", "carmen" in (derivations("cano", "v") or []))
print("   fugio -> fuga  :", "fuga" in (derivations("fugio", "v") or []))
print()
print("The docstring's own example, which it says cannot be found:")
print("   amo -> amor    :", "amor" in (derivations("amo", "v") or []))
print()
print("Coverage is still partial — some lemmas have no entry at all:")
print("   profugus (a)   :", derivations("profugus", "a") or "no lemma entry")

cano     (v) -> ['bucina', 'canor', 'canto', 'cantus', 'carmen', 'concino', 'incino']


fugio    (v) -> ['confugio', 'defugio', 'diffugio', 'fuga', 'fugacius', 'fugax', 'fugela']


bellum   (n) -> ['bellicrepus', 'bellicus', 'belliger', 'belligeratio', 'belligerator', 'belligero', 'bellipotens']


amo      (v) -> ['adamo', 'amans', 'amanter', 'amata', 'amatio', 'amator', 'amatorie']

The poetically relevant one:


   cano -> carmen : True


   fugio -> fuga  : True

The docstring's own example, which it says cannot be found:


   amo -> amor    : True

Coverage is still partial — some lemmas have no entry at all:


   profugus (a)   : no lemma entry


# Part 2 — Which operations can CLTK help with?

Going through the inventory. The question for each is narrow: **does CLTK
provide what the operation's `requires` field asks for?**

In [14]:
from retexo.operations import OperationRegistry

operation_registry = OperationRegistry.default()
frame = operation_registry.to_frame()
frame[frame["detectable"] == False]

,level,role,cost,detectable,requires
tag,,,,,
FRAME,span,writing,0.500,False,citation_verbs
ADAPT,span,marker,0.000,False,stopwords
SUBST,token,writing,0.800,False,—
FORM,token,writing,0.562,False,—
SENSE,token,writing,0.725,False,—
SPLIT,cardinality,writing,0.750,False,morphology
MERGE,cardinality,writing,0.750,False,morphology


## The six undetectable operations, one by one

| Operation | Needs | Does CLTK provide it? | Verdict |
|---|---|---|---|
| **NE-SUB** | `entities` | **Yes** — `tag_ner("lat", …)`, ~39k proper names, offline | **Unblocked.** Recognition solved; pairing two names still needs work |
| **SPLIT** / **MERGE** | `morphology` | **Partly** — the enclitic splitter is broken (§1.2), but the lemmatizer makes a correct one about ten lines of code | **Enclitic case is easy.** Periphrasis is still out of reach |
| **POS** | `derivation` | **Yes, partly** — Latin WordNet's `derivationally_related_forms()`, reachable via CLTK (see §1.10) | **Was wrongly ruled out.** Even *amo→amor*, the docstring's own counter-example, resolves; coverage is partial |
| **FRAME** | `citation_verbs` | **No** — but this needs a word list (*ait*, *inquit*, *scribit*), not a tool | **Not a CLTK problem.** A hand-built list would do |
| **ADAPT** | `stopwords` | **Yes, but inadequate** — 92 entries, missing common function words (§1.6) | **Use frequency instead**, as we already do |

And the operations that are already detectable, for completeness:

| Operation | Needs | Provided by |
|---|---|---|
| `NOP` | nothing | normalization — ours, extended beyond CLTK's |
| `MORPH` | `morphology` | `LatinBackoffLemmatizer` — **in use today** |
| `SYN` `HYPER` `HYPO` `ANT` | `wordnet` | Exeter Latin WordNet, queried directly |
| `SYN-DIST` | `vectors` | word2vec lemma vectors, not CLTK |
| `DISPERSE` | `clauses` | **sentence** boundaries only; clauses need the Stanza parse |
| `QUOTE` `INS` `DEL` `REORDER` | nothing | read off the alignment |

In [15]:
# Concretely: how much of our seed sentence would each resource cover?
tokens = SENTENCE.split()
lemmas = [lemma for _, lemma in lemmatizer.lemmatize(tokens)]

lemmatized = sum(1 for token, lemma in zip(tokens, lemmas) if lemma and lemma != token)
entities = sum(1 for token in tokens if token in names)
stopped = sum(1 for token in tokens if token.lower() in STOPS)

print(f"tokens                     : {len(tokens)}")
print(f"lemma differs from surface : {lemmatized}")
print(f"recognised as proper names : {entities}")
print(f"in the CLTK stoplist       : {stopped}")

tokens                     : 11
lemma differs from surface : 5
recognised as proper names : 2
in the CLTK stoplist       : 2


# Part 3 — What to take from this

**Three things worth acting on.**

1. **`NE-SUB` can be made detectable now.** The blocker recorded in its
   docstring — "needs a named-entity pass" — is satisfied by a 39,725-name list
   that ships with a dependency we already install. This is the cheapest of the
   six undetectable operations to unblock.

2. **Enclitic `SPLIT` is nearly free, but not from CLTK.** The tokenizer
   advertises enclitic splitting and silently does not do it for `-que`,
   `-ne`, `-ue`, `-ve` (§1.2). Ten lines using the lemmatizer as a
   word-existence oracle does the job. Since `-que` is the most common enclitic
   in Latin verse, this is a real slice of the residual.

3. **The Stanza pipeline is the single biggest unused lever.** It would give
   morphological *analysis* rather than generation — so `MORPH` could say which
   feature changed, not merely that something did — and a dependency parse,
   which is the only credible route to the clause boundaries `DISPERSE` is
   defined over. It needs a model download and a real evaluation of its
   accuracy on literary Latin, which is a task of its own.

**Two things this notebook confirms rather than changes.**

- CLTK's stoplist is too small to support `ADAPT`; the frequency-based approach
  already in `substitution.py` is the right call.
- CLTK's WordNet wrapper is the same Exeter data we already query, so it
  inherits the same ~30% coverage ceiling. Going through CLTK would not improve
  the INS/DEL fraction.

**One correction to the code's own documentation.** `POS` is recorded as
blocked because "no derivational lexicon for Latin has been identified". That is
wrong. Latin WordNet exposes `derivationally_related_forms()`, and it resolves
*cano* to *carmen*, *fugio* to *fuga*, and — pointedly — *amo* to *amor*, the
very pair the docstring offers as proof that this cannot be done. Coverage is
partial (no entry for *profugus*, for instance), so `POS` would join `SYN` in
being limited by the wordnet's reach rather than by the absence of a resource.
The docstring should be corrected regardless of whether we enable the operation.